In [ ]:
# Imports.
import numpy as np;
import xarray as xr;
import matplotlib.pyplot as plt;
import h5_reader_xr as reader;
import phi2D_utilities as utils;
import gysela_utilities as gysela_utils;
from matplotlib.gridspec import GridSpec

# Styling.
plt.style.use("ggplot");

In [ ]:
# Test input - delete when done!
directory_path = "PLACEHOLDER";
phi2D_directory_path = f"{directory_path}/sp0/Phi2D/Phi2D_d00000.h5";
magnet_config_path = f"{directory_path}/sp0/init_state/magnet_config_r000.h5";
jacobian_arrays = reader.fetch_jacobian(magnet_config_path);
dataset = reader.fetch_data_from_h5(phi2D_directory_path, dimensions = ["zeta", "r", "theta"]);

In [ ]:
def generate_phi_dictionary(phi2D_dataset):

	phi_FS_avg = gysela_utils.flux_surface_average_2D(phi2D_dataset["Phirth"], jacobian_arrays);
	phi_rth_minus_phi_FS_avg = phi2D_dataset["Phirth"] - phi_FS_avg;
	phi_rth_minus_phi_n0 = phi2D_dataset["Phirth"] - phi2D_dataset["Phirth_n0"];
	return {
		"broadband": {"data": phi2D_dataset["Phirth"], "title": r"Total Potential ($\Phi_{total}$)"},
		"zonal": {"data": phi2D_dataset["Phirth_n0"], "title": r"Zonal Potential ($\Phi_{n=0}$)"},
		"non-zonal": {"data": phi_rth_minus_phi_FS_avg, "title": r"Non-Zonal Potential ($\Phi_{total} - \Phi_{00}$)"},
		"turbulence": {"data": phi_rth_minus_phi_n0, "title": r"Turbulence Potential ($\Phi_{total} - \Phi_{n=0}$)"}
	};

def plot_phi2D(mesh_grid, phi_dictionary, title_suffix):

	x, y = mesh_grid;
	fig = plt.figure(figsize = (12, 10));
	grid = GridSpec(2, 2, figure = fig, wspace = 0.3, hspace = 0.3);
	fig.suptitle(f"Phi2D, {title_suffix}", fontsize = 14);
	axes = [
		fig.add_subplot(grid[0, 0]),
		fig.add_subplot(grid[0, 1]),
		fig.add_subplot(grid[1, 0]),
		fig.add_subplot(grid[1, 1])
	];

	for index, dictionary in enumerate(phi_dictionary.values()):
	
		ax = axes[index];
		phi_values = dictionary["data"].values;
		title = dictionary["title"];

		vmax = np.abs(phi_values).max();
		mesh = ax.pcolormesh(x, y, phi_values, vmax = vmax, vmin = -vmax, shading = "auto", cmap = "bwr");
		ax.contour(x, y, phi_values, levels = 8, colors = "k", linewidths = 0.1, alpha = 0.3);
		ax.set_aspect("equal");
		ax.set_title(title);
		ax.axis("off");
		fig.colorbar(mesh, ax = ax, fraction = 0.046, pad = 0.04);

	plt.show();

phi_data = generate_phi_dictionary(dataset);
mesh_grid = utils.generate_xy_grid(dataset);
plot_phi2D(mesh_grid, phi_data, "PLACEHOLDER");